# Notebook 07a — Video 4 Short-Tail Trajectory Overlay

**Objective:** track the single fish in `4.mp4` with the selected ByteTrack B15 configuration and render a short, time-based bbox-center trajectory tail for observation.

Biological context: this video covers a reproductive period in which the fish may enter the shelter to care for eggs. A disappearing trajectory during shelter entry or egg-care periods may reflect true temporary non-visibility, not necessarily tracker failure. This notebook does not infer breeding events, classify behavior, interpolate missing positions, benchmark another tracker, or start Notebook 08.

## 1. Experiment metadata and CONFIG

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, platform, subprocess, sys, time
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'

EXPERIMENT_ID = 'FRONT_VIDEO4_TRAJECTORY_1S_001'
VIDEO_PATH = PROJECT_ROOT / 'data' / 'raw' / 'front' / '4.mp4'
MODEL_PATH = PROJECT_ROOT / 'runs' / 'front' / 'yolov8n_front_v1_baseline' / 'weights' / 'best.pt'
TRACKER_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'trackers' / 'front_bytetrack_b15.yaml'
DETECTION_CONF = 0.50
TRACK_HIGH_CONF = 0.68
TRAJECTORY_WINDOW_SEC = 1.0
IMGSZ = 640
DEVICE = 0
USE_BYTE_TRACK_B15 = True
EXPECTED_FISH_COUNT = 1
NMS_IOU = 0.70
EXPECTED_MODEL_SHA256 = '750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738'
SOURCE_DETECTION_ID = 'FRONT_VIDEO_DET_CONF068_N1_001'
PROGRESS_INTERVAL = 200
FLOAT_TOLERANCE = 1e-6
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'front' / 'trajectory' / 'video4'
OVERLAY_PATH = OUTPUT_DIR / 'video4_tracking_overlay_1s.mp4'
FRAMEWISE_PATH = OUTPUT_DIR / 'video4_tracking_framewise.csv'
LOG_DIR = PROJECT_ROOT / 'logs' / 'tracking' / EXPERIMENT_ID
RESULT_PATH = PROJECT_ROOT / 'results' / 'tracking' / 'front_video4_trajectory_summary.csv'
SOURCE_CONFIG_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_DETECTION_ID / 'config.yaml'
SOURCE_SUMMARY_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_DETECTION_ID / 'summary.json'

CONFIG = {'experiment_id': EXPERIMENT_ID, 'video_path': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'model_path': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'tracker_config_path': str(TRACKER_CONFIG_PATH.relative_to(PROJECT_ROOT)), 'detection_conf': DETECTION_CONF, 'track_high_conf': TRACK_HIGH_CONF, 'trajectory_window_sec': TRAJECTORY_WINDOW_SEC, 'imgsz': IMGSZ, 'device': DEVICE, 'use_bytetrack_b15': USE_BYTE_TRACK_B15, 'expected_fish_count': EXPECTED_FISH_COUNT, 'nms_iou': NMS_IOU, 'output_dir': str(OUTPUT_DIR.relative_to(PROJECT_ROOT))}
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}; Torch: {torch.__version__}; Ultralytics: {ultralytics.__version__}')
print(f'Device: {DEVICE}; CUDA available: {CUDA_AVAILABLE}; GPU: {GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')
print('CONFIG — VIDEO 4 TRAJECTORY OVERLAY')
for key, value in CONFIG.items(): print(f'{key}: {value}')
if importlib.util.find_spec('lap') is None: raise ModuleNotFoundError('MISSING_DEPENDENCY: ByteTrack requires lap>=0.5.12 in Conda env fish.')

experiment_id: FRONT_VIDEO4_TRAJECTORY_1S_001
datetime_utc: 2026-08-17T12:21:05.254989+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda environment: fish
Python: 3.11.15; Torch: 2.13.0+cu130; Ultralytics: 8.4.120
Device: 0; CUDA available: True; GPU: NVIDIA GeForce RTX 3050
Git commit: de56549b8f773395b1c3fdf0d7841e6559945030
CONFIG — VIDEO 4 TRAJECTORY OVERLAY
experiment_id: FRONT_VIDEO4_TRAJECTORY_1S_001
video_path: data/raw/front/4.mp4
model_path: runs/front/yolov8n_front_v1_baseline/weights/best.pt
tracker_config_path: configs/trackers/front_bytetrack_b15.yaml
detection_conf: 0.5
track_high_conf: 0.68
trajectory_window_sec: 1.0
imgsz: 640
device: 0
use_bytetrack_b15: True
expected_fish_count: 1
nms_iou: 0.7
output_dir: outputs/front/trajectory/video4


## 2. Provenance and ByteTrack B15 preflight

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

assert CONDA_ENV == 'fish', f'FAIL preflight: expected Conda env fish, found {CONDA_ENV!r}'
assert CUDA_AVAILABLE, 'FAIL preflight: CUDA is required because DEVICE=0.'
assert USE_BYTE_TRACK_B15 is True, 'FAIL config: this notebook is restricted to ByteTrack B15.'
assert 0.0 < DETECTION_CONF <= TRACK_HIGH_CONF <= 1.0
assert TRAJECTORY_WINDOW_SEC > 0.0
for required in (VIDEO_PATH, MODEL_PATH, TRACKER_CONFIG_PATH, SOURCE_CONFIG_PATH, SOURCE_SUMMARY_PATH):
    assert required.is_file(), f'FAIL preflight: missing {required.relative_to(PROJECT_ROOT)}'
MODEL_SHA256 = sha256_file(MODEL_PATH)
assert MODEL_SHA256 == EXPECTED_MODEL_SHA256, f'FAIL provenance: model SHA-256 mismatch: {MODEL_SHA256}'
SOURCE_CONFIG = yaml.safe_load(SOURCE_CONFIG_PATH.read_text(encoding='utf-8'))
SOURCE_SUMMARY = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
VIDEO_SHA256 = sha256_file(VIDEO_PATH)
assert SOURCE_CONFIG['experiment_id'] == SOURCE_DETECTION_ID
assert VIDEO_PATH == PROJECT_ROOT / SOURCE_CONFIG['video_path'], 'FAIL provenance: selected video path differs from Notebook 05.'
assert VIDEO_SHA256 == SOURCE_CONFIG['video_sha256'] == SOURCE_SUMMARY['video_sha256'], 'FAIL provenance: video SHA-256 mismatch.'
assert MODEL_SHA256 == SOURCE_SUMMARY['model_sha256']
assert np.isclose(float(SOURCE_CONFIG['nms_iou']), NMS_IOU) and int(SOURCE_CONFIG['imgsz']) == IMGSZ
TRACKER_CONFIG = yaml.safe_load(TRACKER_CONFIG_PATH.read_text(encoding='utf-8'))
assert TRACKER_CONFIG['tracker_type'] == 'bytetrack'
assert np.isclose(float(TRACKER_CONFIG['track_low_thresh']), DETECTION_CONF)
assert np.isclose(float(TRACKER_CONFIG['track_high_thresh']), TRACK_HIGH_CONF)
assert np.isclose(float(TRACKER_CONFIG['new_track_thresh']), TRACK_HIGH_CONF)
assert int(TRACKER_CONFIG['track_buffer']) == 15
assert 0.0 <= float(TRACKER_CONFIG['track_low_thresh']) <= float(TRACKER_CONFIG['track_high_thresh']) <= 1.0
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened(): raise RuntimeError(f'FAIL preflight: cannot open {VIDEO_PATH}')
VIDEO_FPS = float(capture.get(cv2.CAP_PROP_FPS)); TOTAL_FRAMES = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
VIDEO_WIDTH = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)); VIDEO_HEIGHT = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)); capture.release()
assert VIDEO_FPS > 0 and TOTAL_FRAMES > 0 and VIDEO_WIDTH > 0 and VIDEO_HEIGHT > 0
assert TOTAL_FRAMES == int(SOURCE_CONFIG['video_frame_count']) and np.isclose(VIDEO_FPS, float(SOURCE_CONFIG['video_fps']))
WINDOW_FRAMES = max(1, int(round(VIDEO_FPS * TRAJECTORY_WINDOW_SEC)))
DURATION_SEC = TOTAL_FRAMES / VIDEO_FPS
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()): raise RuntimeError(f'FAIL preflight: preserve existing output before rerun: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}')
if RESULT_PATH.exists() or (LOG_DIR.exists() and any(LOG_DIR.iterdir())): raise RuntimeError('FAIL preflight: preserve existing Notebook 07a evidence before rerun.')
print(f'Video: {VIDEO_PATH.relative_to(PROJECT_ROOT)}; SHA-256={VIDEO_SHA256}')
print(f'Frames={TOTAL_FRAMES}; FPS={VIDEO_FPS:.6f}; duration={DURATION_SEC:.3f}s; resolution={VIDEO_WIDTH}x{VIDEO_HEIGHT}')
print(f'Trajectory window: {TRAJECTORY_WINDOW_SEC:.3f}s; logged equivalent WINDOW_FRAMES={WINDOW_FRAMES}')
print(f'ByteTrack B15 config: {TRACKER_CONFIG}')
print('PREFLIGHT_RESULT: PASS')

Video: data/raw/front/4.mp4; SHA-256=3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
Frames=3431; FPS=28.668432; duration=119.679s; resolution=1280x960
Trajectory window: 1.000s; logged equivalent WINDOW_FRAMES=29
ByteTrack B15 config: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
PREFLIGHT_RESULT: PASS


## 3. Track and render the time-based trajectory tail

Each trajectory point is `(time_sec, cx, cy)` keyed by `track_id`. At every frame, points older than `current_time_sec - TRAJECTORY_WINDOW_SEC` are removed. Frames without an active track add no point. After a sufficiently long non-visible interval, the tail becomes empty, so a later observation begins a new tail without interpolation or a line across the shelter period.

In [3]:
from ultralytics import YOLO

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(MODEL_PATH), task='detect')
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened(): raise RuntimeError('FAIL: cannot reopen video for tracking.')
writer = cv2.VideoWriter(str(OVERLAY_PATH), cv2.VideoWriter_fourcc(*'mp4v'), VIDEO_FPS, (VIDEO_WIDTH, VIDEO_HEIGHT))
if not writer.isOpened(): capture.release(); raise RuntimeError(f'FAIL: cannot create {OVERLAY_PATH}')
trajectory_points = {}; frame_rows = []; frame_index = 0; tracker_checked = False; tracking_start = time.perf_counter()
try:
    while True:
        ok, frame = capture.read()
        if not ok: break
        current_time_sec = frame_index / VIDEO_FPS
        result = model.track(source=frame, persist=True, tracker=str(TRACKER_CONFIG_PATH), conf=DETECTION_CONF, iou=NMS_IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        if not tracker_checked:
            actual_tracker = model.predictor.trackers[0]
            ACTUAL_TRACKER_CONFIG = {field: getattr(actual_tracker.args, field) for field in ('tracker_type', 'track_high_thresh', 'track_low_thresh', 'new_track_thresh', 'track_buffer', 'match_thresh', 'fuse_score')}
            assert ACTUAL_TRACKER_CONFIG['tracker_type'] == 'bytetrack' and int(ACTUAL_TRACKER_CONFIG['track_buffer']) == TRACKER_CONFIG['track_buffer']
            for field in ('track_high_thresh', 'track_low_thresh', 'new_track_thresh', 'match_thresh'):
                assert np.isclose(float(ACTUAL_TRACKER_CONFIG[field]), float(TRACKER_CONFIG[field])), f'FAIL actual tracker config: {field}'
            print(f'ACTUAL TRACKER CONFIG: {ACTUAL_TRACKER_CONFIG}'); tracker_checked = True
        cutoff_time_sec = current_time_sec - TRAJECTORY_WINDOW_SEC
        for track_id in list(trajectory_points):
            trajectory_points[track_id] = [point for point in trajectory_points[track_id] if point[0] >= cutoff_time_sec]
            if not trajectory_points[track_id]: del trajectory_points[track_id]
        boxes = result.boxes
        observed = boxes is not None and boxes.id is not None and len(boxes.id) > 0
        overlay = frame.copy()
        if observed:
            track_ids = boxes.id.detach().cpu().numpy().astype(int)
            confidences = boxes.conf.detach().cpu().numpy().astype(float)
            xyxy = boxes.xyxy.detach().cpu().numpy().astype(float)
            assert float(confidences.min()) >= DETECTION_CONF - FLOAT_TOLERANCE, 'FAIL: detection below configured floor entered tracker output.'
            for track_id, confidence, coords in zip(track_ids, confidences, xyxy):
                x1, y1, x2, y2 = coords.tolist(); cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                trajectory_points.setdefault(int(track_id), []).append((current_time_sec, cx, cy))
                frame_rows.append({'frame_index': frame_index, 'time_sec': current_time_sec, 'track_id': int(track_id), 'confidence': float(confidence), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'cx': cx, 'cy': cy, 'track_present': 1})
                p1, p2 = (int(round(x1)), int(round(y1))), (int(round(x2)), int(round(y2)))
                cv2.rectangle(overlay, p1, p2, (0, 220, 0), 2)
                cv2.circle(overlay, (int(round(cx)), int(round(cy))), 5, (0, 0, 255), -1)
                cv2.putText(overlay, f'ID {track_id} | {confidence:.2f}', (p1[0], max(18, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
        else:
            frame_rows.append({'frame_index': frame_index, 'time_sec': current_time_sec, 'track_id': np.nan, 'confidence': np.nan, 'x1': np.nan, 'y1': np.nan, 'x2': np.nan, 'y2': np.nan, 'cx': np.nan, 'cy': np.nan, 'track_present': 0})
        for track_id, points in trajectory_points.items():
            if len(points) >= 2:
                polyline = np.asarray([(int(round(point[1])), int(round(point[2]))) for point in points], dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(overlay, [polyline], False, (0, 180, 255), 3, cv2.LINE_AA)
        active_count = int(boxes.id.numel()) if observed else 0
        header = f'frame={frame_index}/{TOTAL_FRAMES - 1} | t={current_time_sec:.2f}s | active_tracks={active_count} | trajectory window={TRAJECTORY_WINDOW_SEC:.1f}s'
        cv2.rectangle(overlay, (0, 0), (min(VIDEO_WIDTH, 1050), 36), (0, 0, 0), -1)
        cv2.putText(overlay, header, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.64, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(overlay); frame_index += 1
        if frame_index % PROGRESS_INTERVAL == 0 or frame_index == TOTAL_FRAMES:
            elapsed = time.perf_counter() - tracking_start
            print(f'Processed {frame_index}/{TOTAL_FRAMES} | elapsed={elapsed:.1f}s | processing FPS={frame_index / elapsed:.2f}')
finally:
    capture.release(); writer.release()
TRACKING_RUNTIME_SEC = time.perf_counter() - tracking_start
assert frame_index == TOTAL_FRAMES, f'FAIL: incomplete tracking {frame_index}/{TOTAL_FRAMES}'
assert tracker_checked, 'FAIL: actual tracker configuration was not inspected.'
FRAMEWISE_DF = pd.DataFrame(frame_rows, columns=['frame_index', 'time_sec', 'track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy', 'track_present'])
assert FRAMEWISE_DF['frame_index'].nunique() == TOTAL_FRAMES
FRAMEWISE_DF.to_csv(FRAMEWISE_PATH, index=False)
print(f'Created {OVERLAY_PATH.relative_to(PROJECT_ROOT)} ({OVERLAY_PATH.stat().st_size} bytes)')
print(f'Created {FRAMEWISE_PATH.relative_to(PROJECT_ROOT)} ({len(FRAMEWISE_DF)} rows; {FRAMEWISE_PATH.stat().st_size} bytes)')

ACTUAL TRACKER CONFIG: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
Processed 200/3431 | elapsed=6.4s | processing FPS=31.34
Processed 400/3431 | elapsed=10.8s | processing FPS=37.16
Processed 600/3431 | elapsed=15.5s | processing FPS=38.78
Processed 800/3431 | elapsed=19.8s | processing FPS=40.38
Processed 1000/3431 | elapsed=23.8s | processing FPS=41.94
Processed 1200/3431 | elapsed=27.5s | processing FPS=43.58
Processed 1400/3431 | elapsed=31.2s | processing FPS=44.87
Processed 1600/3431 | elapsed=34.7s | processing FPS=46.09
Processed 1800/3431 | elapsed=38.2s | processing FPS=47.15
Processed 2000/3431 | elapsed=41.7s | processing FPS=47.95
Processed 2200/3431 | elapsed=45.8s | processing FPS=48.04
Processed 2400/3431 | elapsed=49.9s | processing FPS=48.09
Processed 2600/3431 | elapsed=53.9s | processing FPS=48.21
Processed 2800/3431 | elapsed=58.0s | processi

## 4. Summary evidence and cautious interpretation

In [4]:
FRAME_PRESENT = FRAMEWISE_DF.groupby('frame_index')['track_present'].max().reindex(np.arange(TOTAL_FRAMES), fill_value=0).astype(int)
TRACKED_FRAMES = int((FRAME_PRESENT == 1).sum()); UNTRACKED_FRAMES = int((FRAME_PRESENT == 0).sum()); TRACKED_RATE = TRACKED_FRAMES / TOTAL_FRAMES
WARNINGS = []
if UNTRACKED_FRAMES > 0: WARNINGS.append('Some frames have no active observed track. During shelter or egg-care periods this may reflect true temporary non-visibility, not necessarily tracker failure.')
CHECKPOINT_RESULT = 'PASS_WITH_WARNING' if WARNINGS else 'PASS'
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True); LOG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOG_DIR / 'config.yaml'; ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'; SUMMARY_PATH = LOG_DIR / 'summary.json'
SUMMARY_ROW = {'experiment_id': EXPERIMENT_ID, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'model_sha256': MODEL_SHA256, 'tracker': str(TRACKER_CONFIG_PATH.relative_to(PROJECT_ROOT)), 'trajectory_window_sec': TRAJECTORY_WINDOW_SEC, 'window_frames': WINDOW_FRAMES, 'frames': TOTAL_FRAMES, 'fps': VIDEO_FPS, 'duration_sec': DURATION_SEC, 'tracked_frames': TRACKED_FRAMES, 'untracked_frames': UNTRACKED_FRAMES, 'tracked_rate': TRACKED_RATE, 'processing_FPS': TOTAL_FRAMES / TRACKING_RUNTIME_SEC, 'checkpoint_result': CHECKPOINT_RESULT}
pd.DataFrame([SUMMARY_ROW]).to_csv(RESULT_PATH, index=False)
CONFIG_EVIDENCE = {**CONFIG, 'video_sha256': VIDEO_SHA256, 'model_sha256': MODEL_SHA256, 'video_fps': VIDEO_FPS, 'total_frames': TOTAL_FRAMES, 'window_frames': WINDOW_FRAMES, 'requested_tracker_config': TRACKER_CONFIG, 'actual_tracker_config': ACTUAL_TRACKER_CONFIG, 'git_commit': GIT_COMMIT}
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_runtime={torch.version.cuda}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}', f'opencv={cv2.__version__}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
OUTPUT_FILES = [OVERLAY_PATH, FRAMEWISE_PATH, RESULT_PATH, CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH]
SUMMARY = {**SUMMARY_ROW, 'context_note': 'Trajectory disappearance during shelter entry or egg-care periods may reflect true temporary non-visibility; no behavior/event label is inferred.', 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'git_commit': GIT_COMMIT, 'next_step': 'USER reviews the Video 4 trajectory overlay; do not start Notebook 08 without explicit approval.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for path in (RESULT_PATH, CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH): print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

Created results/tracking/front_video4_trajectory_summary.csv (589 bytes)
Created logs/tracking/FRONT_VIDEO4_TRAJECTORY_1S_001/config.yaml (1045 bytes)
Created logs/tracking/FRONT_VIDEO4_TRAJECTORY_1S_001/environment.txt (356 bytes)
Created logs/tracking/FRONT_VIDEO4_TRAJECTORY_1S_001/summary.json (1649 bytes)


## 5. Final Summary

In [5]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'tracker': str(TRACKER_CONFIG_PATH.relative_to(PROJECT_ROOT)), 'trajectory_window_sec': TRAJECTORY_WINDOW_SEC, 'window_frames': WINDOW_FRAMES, 'frames': TOTAL_FRAMES, 'fps': VIDEO_FPS, 'duration_sec': DURATION_SEC, 'tracked_frames': TRACKED_FRAMES, 'untracked_frames': UNTRACKED_FRAMES, 'tracked_rate': TRACKED_RATE, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': WARNINGS, 'next_step': SUMMARY['next_step']}
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_VIDEO4_TRAJECTORY_1S_001
video: data/raw/front/4.mp4
model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
tracker: configs/trackers/front_bytetrack_b15.yaml
trajectory_window_sec: 1.0
window_frames: 29
frames: 3431
fps: 28.66843196767745
duration_sec: 119.67867666666668
tracked_frames: 2380
untracked_frames: 1051
tracked_rate: 0.6936753133197319
output_files: ['outputs/front/trajectory/video4/video4_tracking_overlay_1s.mp4', 'outputs/front/trajectory/video4/video4_tracking_framewise.csv', 'results/tracking/front_video4_trajectory_summary.csv', 'logs/tracking/FRONT_VIDEO4_TRAJECTORY_1S_001/config.yaml', 'logs/tracking/FRONT_VIDEO4_TRAJECTORY_1S_001/environment.txt', 'logs/tracking/FRONT_VIDEO4_TRAJECTORY_1S_001/summary.json']
checkpoint_result: PASS_WITH_WARNING
warnings: ['Some frames have no active observed track. During shelter or egg-care periods this may reflect true temporary non-visibility, not necessarily tracker failure.']
next_step: US